# Complementary Error Analysis — Vision vs Text vs Ground Truth

The dual-expert claim rests on the two experts making **different** mistakes. This
notebook finds and characterizes those complementary cases on both datasets. For every
ground-truth cell it compares what the vision expert, the text expert, and the ground
truth say

In [ ]:
from pathlib import Path
import json, re
from collections import Counter
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import Levenshtein

DATASETS = {
    "A25": {
        "gt_root":     Path("data/A25/input_images"),
        "vision_root": Path("data/outputs/vision_expert_gemma4_run"),
        "text_root":   Path("data/outputs/text_expert_gemma4_run"),
        "groups":      ["Biology", "CompSci", "ICDAR", "MatSci"],  
        "gt_subdir": "xmls", 
        "vision_subdir": "predictions", 
        "text_subdir": "nougat/predictions",
        "gt_format": "xml",
    },
    "SciTSR": {
        "gt_root":     Path("data/SciTSR/"),
        "vision_root": Path("data/outputs/vision_expert_gemma4_SciTSR_run"),
        "text_root":   Path("data/outputs/text_expert_gemma4_SciTSR_run"),
        "groups":      ["test"],                                    
        "gt_subdir": "structure_processed", 
        "vision_subdir": "predictions", 
        "text_subdir": "predictions",
        "gt_format": "json",
    },
}
LEV_THRESHOLD = 1.0  
N_EXAMPLES    = 20     # examples shown per difference category

## Step 1 — Similar functionalities for calculating similarities

Identical to the router notebooks.

In [ ]:
def normalize_text(text):
    if text is None:

        return ""
    
    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")
    return re.sub(r"\s+", " ", text).strip()

def lev_sim(a, b):

    a, b = normalize_text(a), normalize_text(b)

    if a == "" and b == "":
        return 1.0
    
    return 1.0 - Levenshtein.distance(a, b) / max(len(a), len(b), 1)

def cell_key(c):

    return (int(c["sr"]), int(c["er"]), int(c["sc"]), int(c["ec"]))

def parse_gt_xml(path):

    if path is None or not Path(path).exists():
        return []
    
    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.findall("cell"):
        sr, sc = c.get("start_row"), c.get("start_col")

        if sr is None or sc is None:
            continue

        er, ec = c.get("end_row", sr), c.get("end_col", sc)
        t = c.find("text")

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text((t.text or "") if t is not None else "")
        })

    return out

def _scitsr_gt_text(cell):
    txt = cell.get("text")

    if not txt:
        content = cell.get("content")
        txt = " ".join(str(t) for t in content) if isinstance(content, list) else (str(content) if content else "")

    if not txt:
        txt = cell.get("tex", "")

    return normalize_text(txt)

def parse_gt_json(path):

    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data
    out = []
    for c in cells:
        sr, sc = c.get("start_row", c.get("sr")), c.get("start_col", c.get("sc"))

        if sr is None or sc is None:
            continue

        er, ec = c.get("end_row", c.get("er", sr)), c.get("end_col", c.get("ec", sc))
        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": _scitsr_gt_text(c)
        })

    return out

def parse_pred_json(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data
    if not isinstance(cells, list):
        return []
    
    out = []
    for c in cells:

        if not isinstance(c, dict):
            continue

        sr, sc = c.get("sr", c.get("start_row")), c.get("sc", c.get("start_col"))

        if sr is None or sc is None:
            continue

        er, ec = c.get("er", c.get("end_row", sr)), c.get("ec", c.get("end_col", sc))
        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text(c.get("text", ""))
        })

    return out

def pred_lookup(cells_):
    out = {}

    for c in cells_:
        out.setdefault(cell_key(c), c)

    return out

def discover(ds):
    cfg, rows = DATASETS[ds], []
    for grp in cfg["groups"]:

        gt_dir  = cfg["gt_root"] / grp / cfg["gt_subdir"]
        vis_dir = cfg["vision_root"] / grp / cfg["vision_subdir"]
        txt_dir = cfg["text_root"] / grp / cfg["text_subdir"]
        ext = "*.xml" if cfg["gt_format"] == "xml" else "*.json"

        gt  = {p.stem: p for p in sorted(gt_dir.glob(ext))} if gt_dir.exists() else {}
        vis = {p.stem: p for p in vis_dir.glob("*.json")} if vis_dir.exists() else {}
        txt = {p.stem: p for p in txt_dir.glob("*.json")} if txt_dir.exists() else {}

        for s, g in gt.items():
            if s in vis and s in txt:
                rows.append({
                    "group": grp, 
                    "stem": s, 
                    "table_id": f"{grp}::{s}",
                    "gt_file": g, 
                    "vision_file": vis[s], 
                    "text_file": txt[s]
                })

    return pd.DataFrame(rows)

tables = {ds: discover(ds) for ds in DATASETS}
for ds, df in tables.items():
    print(f"{ds}: {len(df)} tables with both expert predictions")

## Step 2 — The symbolic/semantic classifier

In [ ]:
# --- Symbolic vs semantic classification --------------------------------------
import unicodedata

LATEX_CMD_RE = re.compile(r"\\[a-zA-Z]+")

def _latex_cmd(m):

    name = m.group(0)[1:]

    if name == "pm":
        return "±"
    
    if name == "lambda":
        name = "lamda"  

    try:
        return unicodedata.lookup(f"GREEK SMALL LETTER {name.upper()}")
    except KeyError:
        return " "

def semantic_skeleton(t):

    s = re.sub(r"(?<=[\d\s])pm(?=[\s\d])", "±", t)     
    s = LATEX_CMD_RE.sub(_latex_cmd, s)
    s = unicodedata.normalize("NFKC", s)                
    s = s.replace("−", "-")                        
    s = s.replace("×", "x").replace("·", " ").replace("∙", " ")
    s = re.sub(r"[{}\\$^_]", "", s)

    return re.sub(r"\s+", " ", s).strip()

def digits_of(s):
    return re.sub(r"\D", "", s)

def diff_category(a, b):
    if a == b:
        return "identical"

    if a == "" or b == "":
        return "empty content"

    sa, sb = semantic_skeleton(a), semantic_skeleton(b)
    if sa == sb or sa.replace(" ", "") == sb.replace(" ", ""):
        return "symbolic"

    if sa and sb and (sa in sb or sb in sa):
        return "truncation / merge"

   
    if (re.search(r"\d", sa) and re.search(r"\d", sb)
            and digits_of(sa) != digits_of(sb)
            and re.sub(r"[\d\s.,]", "", sa) == re.sub(r"[\d\s.,]", "", sb)):
        return "numeric value"

    return "other semantic"

def expert_vs_gt(expert_cell, gt_text):
    if expert_cell is None:
        return "missing cell"

    cat = diff_category(normalize_text(expert_cell.get("text", "")), gt_text)

    return "correct" if cat == "identical" else cat


## Step 3 — One row per ground-truth cell, fully labeled

In [ ]:

def build_analysis(ds):

    cfg = DATASETS[ds]
    gt_parse = parse_gt_xml if cfg["gt_format"] == "xml" else parse_gt_json

    rows = []
    for _, tab in tables[ds].iterrows():

        gt = gt_parse(tab["gt_file"])
        vl = pred_lookup(parse_pred_json(tab["vision_file"]))
        tl = pred_lookup(parse_pred_json(tab["text_file"]))

        for g in gt:

            k = cell_key(g)
            vc, tc = vl.get(k), tl.get(k)

            vt = "" if vc is None else vc["text"]
            tt = "" if tc is None else tc["text"]
            v_ok = vc is not None and lev_sim(g["text"], vt) >= LEV_THRESHOLD
            t_ok = tc is not None and lev_sim(g["text"], tt) >= LEV_THRESHOLD

            rows.append({
                "dataset": ds, 
                "group": tab["group"], 
                "table_id": tab["table_id"],
                "gt_text": g["text"], 
                "vision_text": vt, 
                "text_text": tt,
                "vision_present": vc is not None, 
                "text_present": tc is not None,
                "vision_correct": bool(v_ok), 
                "text_correct": bool(t_ok),
                "one_correct": bool(v_ok != t_ok),
                "winner": "vision" if (v_ok and not t_ok) else ("text" if (t_ok and not v_ok) else ("both" if v_ok else "neither")),
                "vt_diff": ("missing cell" if (vc is None) != (tc is None) else diff_category(vt, tt)),
                "vision_vs_gt": expert_vs_gt(vc, g["text"]),
                "text_vs_gt":   expert_vs_gt(tc, g["text"]),
            })
    return pd.DataFrame(rows)

df = pd.concat([build_analysis(ds) for ds in DATASETS], ignore_index=True)
print(df.groupby("dataset").agg(cells=("gt_text", "size"),
                                vision_acc=("vision_correct", "mean"),
                                text_acc=("text_correct", "mean"),
                                complementary=("one_correct", "mean")).round(4))

## Step 4 — The complementary examples themselves

Side-by-side GT / Vision / Text for every complementary cell (exactly one expert
correct), exported per dataset to `complementary_examples_<dataset>.csv`. Below, a few
examples per difference type.

In [ ]:

cols = ["dataset", "group", "table_id", "gt_text", "vision_text", "text_text",
        "winner", "vt_diff", "vision_vs_gt", "text_vs_gt"]

for ds in DATASETS:

    comp = df[(df["dataset"] == ds) & (df["one_correct"])][cols]
    out = f"complementary_examples_{ds}.csv"
    comp.to_csv(out, index=False)

    print(f"{ds}: wrote {len(comp)} complementary cells -> {out}")

pd.set_option("display.max_colwidth", 40)
comp_all = df[df["one_correct"]]

for cat in comp_all["vt_diff"].value_counts().index:
    ex = comp_all[comp_all["vt_diff"] == cat]

    print(f"\n {cat} — {len(ex)} complementary cells")
    display(ex[["dataset", "gt_text", "vision_text", "text_text", "winner"]].head(N_EXAMPLES))